<div align="center"><img src="https://github.com/user-attachments/assets/5d3a38ab-b20c-4103-b401-cddc112bfac8" width="300" alt="NOVA IMS Logo">

  
  <b>Master in Data Science and Advanced Analytics</b>

  <h3><b>Extending SLIM-GSGP to Multiclass Classification: An Application to Alzheimer's Disease Diagnosis </b></h3>



  <h2><b> <i>ADNIMERGE</i> Preprocessing <b></h2>
</div>

---

# **Table of Contents**

1. [Data Preparation for Preprocessing](#section-1) <br>
    1.1. [Importing Libraries and Datasets](#section-1-1) <br>
    1.2. [Data Storage](#section-1-2) <br>
2. [Data Preparation](#section-2) <br>
    2.1. [Records Selection (rows)](#section-2-1) <br>
    2.2. [Feature Preparation (columns)](#section-2-2) <br>
    &nbsp;&nbsp;&nbsp;2.2.1. [Type conversion](#section-2-2-1) <br>
    &nbsp;&nbsp;&nbsp;2.2.2. [Incoherencies treatment](#section-2-2-2) <br>
    &nbsp;&nbsp;&nbsp;2.2.3. [Features Exclusion](#section-2-2-3) <br>
    &nbsp;&nbsp;&nbsp;2.2.3. [Categorical Encoding](#section-2-2-4) <br>
    2.3. [Data Partitioning](#section-2-3) <br>
3. [Preprocessing](#section-3) <br>
    3.1. [Missing Values](#section-3-1) <br>
    3.2. [Scaling](#section-3-2) <br>
    3.3. [Encoding](#section-3-3) <br>
    3.4. [Model](#section-3-4) <br>
    3.5. [Feature Engineering](#section-3-5) <br>
    3.6. [Feature Selection](#section-3-6) <br>

<a id="section-1"></a>

# **<font color='#ffffffff'> 1. Data Importation and Integration </font>**

<a id="section-1-1"></a>

## <font color='#0fe073'> 1.1. Importing Libraries and Datasets </font>

In [28]:
# General setup & utilities
import warnings
from math import ceil
import sys

# Data Manipulation & loading
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [2]:
# Data
original_data = pd.read_csv('data/ADNIMERGE.csv', low_memory=False)
# Set copy
adni_data = original_data.copy()

<a id="section-1-2"></a>
    
## <font color='#0fe073'>1.2. Data Storage </font>

In [3]:
print(adni_data.info(memory_usage='deep'))

<class 'pandas.DataFrame'>
RangeIndex: 16421 entries, 0 to 16420
Columns: 116 entries, RID to update_stamp
dtypes: float64(88), int64(5), str(23)
memory usage: 30.9 MB
None


In [4]:
#Reduce memory for faster processing - int32 / float32
def reduce_memory_usage(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            if 'int' in str(df[col].dtype):
                df[col] = df[col].astype('int32')
            elif 'float' in str(df[col].dtype):
                df[col] = df[col].astype('float32')
    return df

In [5]:
adni_data = reduce_memory_usage(adni_data)
adni_data.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 16421 entries, 0 to 16420
Columns: 116 entries, RID to update_stamp
dtypes: float32(88), int32(5), str(23)
memory usage: 25.1 MB


<a id="section-2"></a>

# **<font color='#ffffffff'> 2. Data Preparation </font>**

<a id="section-2-1"></a>

## <font color='#0fe073'> 2.1. Records Selection (rows) </font>


- remover visitas sem DX
- baseline (1 linha por sujeito)

In [6]:
# Baseline selection: 1 line per participant, no longitudinal component
n_subj_all = adni_data["RID"].nunique()

first = (adni_data[adni_data["VISCODE"] == "bl"]
         .dropna(subset=["DX"])
         .sort_values("RID")
         .reset_index(drop=True)
         .copy())

assert first["RID"].is_unique, "There are participants with more than one baseline"

print(f"Visits: {len(adni_data)} → {len(first)}")
print(f"Subjects: {n_subj_all} → {first['RID'].nunique()} "
      f"({n_subj_all - first['RID'].nunique()} baseline without DX)")
print(first["DX"].value_counts().reindex(["CN", "MCI", "Dementia"]).to_string())

Visits: 16421 → 2409
Subjects: 2430 → 2409 (21 baseline without DX)
DX
CN           895
MCI         1101
Dementia     413


In [7]:
# Backup of the baseline cohort
baseline_adni = first.copy()

<a id="section-2-2"></a>

## <font color='#0fe073'> 2.2. Feature Preparation (columns) </font>

- Type conversion        (CSF para numérico + flags)
- Invalid value correction (TRABSCOR, WholeBrain > ICV + verificação)
- Variable exclusion
- Categorical encoding

<a id="section-2-2-1"></a>

### <font color='#0fe073'> 2.2.1. Type conversion </font>

In [8]:
# 1) CSF para numérico, com flag de censura - VOLTAR  A VER
for c in ["ABETA", "TAU", "PTAU"]:
    if not pd.api.types.is_numeric_dtype(baseline_adni[c]):
        raw = baseline_adni[c].astype("string")
        baseline_adni[f"{c}_censored"] = raw.str.contains(r"[<>]", na=False).astype(int)
        baseline_adni[c] = pd.to_numeric(raw.str.replace(r"[<>]", "", regex=True), errors="coerce")

<a id="section-2-2-2"></a>

### <font color='#0fe073'> 2.2.2. Incoherencies treatment </font>

In [9]:
# Correções de valores impossíveis
log = {}
is_bl = baseline_adni["VISCODE"].eq("bl")

1. **`TRABSCOR` CASE**
- `TRABSCOR` is the standardized variable name for the Trail Making Test Part B (TMT-B) completion time in major Alzheimer's disease research datasets, having a span of possible values between 0 and 300
- Violations: 39 participants with times outside the range
- Analysis: 
    - Value 0 (x cases): Its impossible to complete the test in 0 seconds, therefore there is an high chance that the test was not administrated or an eror in regestration occor. **Treatment as NaN**
    - Values 301–510 (11 cases): Time above the test limit, there is an high chance that the participant didn't conclude the test. **Treatment by clip at 300 + flag**
    - Values 996 (2 cases): Possible code of missingness. **Treatment as NaN**

In [10]:
# TRABSCOR: 0 s e 996 → NaN; >300 → clip + flag
t = baseline_adni["TRABSCOR"]
t_nan  = (t == 0) | (t >= 900)
t_clip = (t > 300) & (t < 900)
log["TRABSCOR 0/996 → NaN"] = (int(t_nan.sum()),  int((t_nan & is_bl).sum()))
log["TRABSCOR >300 → 300"]  = (int(t_clip.sum()), int((t_clip & is_bl).sum()))

In [11]:
baseline_adni.loc[t_nan, "TRABSCOR"] = np.nan
baseline_adni["TRABSCOR"] = baseline_adni["TRABSCOR"].clip(upper=300)
baseline_adni["TRABSCOR_capped"] = baseline_adni["TRABSCOR"].eq(300).astype(int)

2. **`WholeBrain` > `ICV` CASE**
- `ICV` is the total cranial cavity (grey matter, white matter, CSF and meninges) and `WholeBrain` is the Parenchymal brain tissue only (excludes CSF). Therefore, having values of WholeBrain superior to ICV is not logically possible.
-  Violations: 2 entries 
- Analysis: **use the participant as reference, based on other correct entries. If the participant only has that entry treat as NaN.**

In [12]:
icv_ref = adni_data.groupby("RID")["ICV"].agg(["median", "count"])   # only use of adni_data
icv_med = baseline_adni["RID"].map(icv_ref["median"])
n_icv   = baseline_adni["RID"].map(icv_ref["count"]).fillna(0)

In [13]:
bad     = baseline_adni["WholeBrain"] > baseline_adni["ICV"]
icv_off = (baseline_adni["ICV"] - icv_med).abs() / icv_med > 0.10   # 10%: above normal variation (~5%)

In [14]:
c1 = bad & (n_icv > 1) & icv_off      # ICV errado → mediana do sujeito
c2 = bad & (n_icv > 1) & ~icv_off     # WholeBrain errado → NaN
c3 = bad & (n_icv <= 1)               # sem referência → os dois NaN
log["WholeBrain>ICV: ICV corrected"] = int(c1.sum())
log["WholeBrain>ICV: WholeBrain → NaN"] = int(c2.sum())
log["WholeBrain>ICV: both → NaN"] = int(c3.sum())

In [15]:
baseline_adni.loc[c1, "ICV"] = icv_med[c1]
baseline_adni.loc[c2, "WholeBrain"] = np.nan
baseline_adni.loc[c3, ["WholeBrain", "ICV"]] = np.nan

**Violations Summary and verification after treatment**

In [16]:
print(pd.Series(log, name="n (baseline)").to_string())

TRABSCOR 0/996 → NaN                (1, 1)
TRABSCOR >300 → 300                 (0, 0)
WholeBrain>ICV: ICV corrected            0
WholeBrain>ICV: WholeBrain → NaN         0
WholeBrain>ICV: both → NaN               0


In [17]:
t = baseline_adni["TRABSCOR"]
checks = {
    "TRABSCOR outside ]0, 300]": int((t.notna() & ~t.between(0, 300, inclusive="right")).sum()),
    "WholeBrain > ICV":          int((baseline_adni["WholeBrain"] > baseline_adni["ICV"]).sum()),
    "TRABSCOR_capped mismatch":  int((baseline_adni["TRABSCOR_capped"] != baseline_adni["TRABSCOR"].eq(300)).sum()),
}
print("\nVerification after treatment:")
print(pd.Series(checks).to_string())
assert sum(checks.values()) == 0, "Invalid values remain after treatment!"


Verification after treatment:
TRABSCOR outside ]0, 300]    0
WholeBrain > ICV             0
TRABSCOR_capped mismatch     0


<a id="section-2-2-3"></a>

### <font color='#0fe073'> 2.2.3. Features Exclusion </font>

In [18]:
# Variable exclusion
drop_groups = {
    "Identifiers / admin": ["COLPROT", "ORIGPROT", "PTID", "SITE",  "VISCODE", "EXAMDATE",
                            "update_stamp", "IMAGEUID"],
    "Time": ["Month", "M"],
    "Technical": ["FLDSTRENG", "FSVERSION"],
    "Demographics": ["PTRACCAT", "PTETHCAT"], 
    "PET (other tracers)": ["PIB", "FBB"],
    "Redundant": ["TAU", "ADAS11", "ADASQ4", "mPACCdigit", "mPACCtrailsB",
                            "RAVLT_perc_forgetting", "TAU_censored"],
    "Baseline copies": [c for c in baseline_adni.columns if c.lower().endswith("_bl")],  # inclui LDELTOTAL_BL
    "Ecog": [c for c in baseline_adni.columns if c.startswith(("EcogPt", "EcogSP"))],
}

summary, drop_cols = [], []
for group, cols in drop_groups.items():
    present = [c for c in cols if c in baseline_adni.columns]
    missing = sorted(set(cols) - set(present))
    drop_cols += present
    summary.append({"group": group, "n": len(present),
                    "missing (not in data)": ", ".join(missing) or "—"})

n_before = baseline_adni.shape[1]
baseline_adni = baseline_adni.drop(columns=sorted(set(drop_cols)))

print(pd.DataFrame(summary).to_string(index=False))
print(f"\nColumns: {n_before} → {baseline_adni.shape[1]}  (dropped {n_before - baseline_adni.shape[1]})")
print("Remaining:", sorted(baseline_adni.columns))

              group  n missing (not in data)
Identifiers / admin  8                     —
               Time  2                     —
          Technical  2                     —
       Demographics  2                     —
PET (other tracers)  2                     —
          Redundant  7                     —
    Baseline copies 51                     —
               Ecog 28                     —

Columns: 120 → 32  (dropped 88)
Remaining: ['ABETA', 'ABETA_censored', 'ADAS13', 'AGE', 'APOE4', 'AV45', 'CDRSB', 'DIGITSCOR', 'DX', 'Entorhinal', 'FAQ', 'FDG', 'Fusiform', 'Hippocampus', 'ICV', 'LDELTOTAL', 'MMSE', 'MOCA', 'MidTemp', 'PTAU', 'PTAU_censored', 'PTEDUCAT', 'PTGENDER', 'PTMARRY', 'RAVLT_forgetting', 'RAVLT_immediate', 'RAVLT_learning', 'RID', 'TRABSCOR', 'TRABSCOR_capped', 'Ventricles', 'WholeBrain']


In [19]:
baseline_adni = baseline_adni.set_index("RID")

<a id="section-2-2-4"></a>

### <font color='#0fe073'> 2.2.4. Categorical Encoding </font>

|   | Feature | Type | Encoding |
|---|---|---|---|
| 1 | DX | Target, 3 classes | Label encoding: CN=0, MCI=1, Dementia=2 |
| 2 | PTGENDER | Binary | Female=0 Male=1 |
| 3 | PTMARRY | Nominal (Married, Widowed, Divorced, Never married, Unknown) | Female=0 Male=1 | One-hot. Treat unknown as missing



In [ ]:
baseline_adni

PTGENDER
Male      1265
Female    1144
Name: count, dtype: int64 

PTMARRY
Married          1812
Widowed           266
Divorced          224
Never married      97
Unknown             9
NaN                 1
Name: count, dtype: int64 

DX
MCI         1101
CN           895
Dementia     413
Name: count, dtype: int64 



In [21]:
# 1) Target
baseline_adni["DX"] = baseline_adni["DX"].map({"CN": 0, "MCI": 1, "Dementia": 2}).astype(int)

In [22]:
# 2) PTGENDER: binária
baseline_adni["PTGENDER"] = baseline_adni["PTGENDER"].map({"Female": 0, "Male": 1})

In [23]:
# 3) PTMARRY: "Unknown" -> NaN (imputa-se depois), depois one-hot
baseline_adni["PTMARRY"] = baseline_adni["PTMARRY"].replace("Unknown", np.nan)
baseline_adni["PTMARRY"] = pd.Categorical(
    baseline_adni["PTMARRY"],
    categories=["Married", "Widowed", "Divorced", "Never married"]
)

marry_na = baseline_adni["PTMARRY"].isna()
dummies = pd.get_dummies(baseline_adni["PTMARRY"], prefix="PTMARRY", drop_first=True, dtype=float)
dummies.loc[marry_na, :] = np.nan
baseline_adni = pd.concat([baseline_adni.drop(columns="PTMARRY"), dummies], axis=1)

In [ ]:
# Verification
print(baseline_adni["PTGENDER"].isna().sum()) # 0
print(baseline_adni["PTGENDER"].value_counts()) # 1: 1265, 0: 1144
print(baseline_adni["DX"].value_counts()) # 1: 1101, 0: 895, 2: 413
print(baseline_adni.filter(like="PTMARRY").sum()) # Married: 1812, Never married: 97, Widowed: 266, Divorced; 224, 
print(baseline_adni.filter(like="PTMARRY").isna().sum()) # 10 in each column

0
PTGENDER
1    1265
0    1144
Name: count, dtype: int64
DX
1    1101
0     895
2     413
Name: count, dtype: int64
PTMARRY_Widowed          266.0
PTMARRY_Divorced         224.0
PTMARRY_Never married     97.0
dtype: float64
PTMARRY_Widowed          10
PTMARRY_Divorced         10
PTMARRY_Never married    10
dtype: int64


---

<a id="section-2-3"></a>

## <font color='#0fe073'> 2.3 Data Partitioning  </font>

In [30]:
baseline_adni

,AGE,PTGENDER,PTEDUCAT,APOE4,FDG,AV45,ABETA,PTAU,CDRSB,ADAS13,...,Fusiform,MidTemp,ICV,DX,ABETA_censored,PTAU_censored,TRABSCOR_capped,PTMARRY_Widowed,PTMARRY_Divorced,PTMARRY_Never married
RID,,,,,,,,,,,,,,,,,,,,,
2,74.300003,1,16,0.0,1.33615,NaN,<NA>,<NA>,0.0,18.67,...,16559.0,27936.0,1984660.0,0,0,0,0,0.0,0.0,0.0
3,81.300003,1,18,1.0,1.10860,NaN,741.5,22.83,4.5,31.00,...,15506.0,18422.0,1920690.0,2,0,0,0,0.0,0.0,0.0
4,67.500000,1,10,0.0,NaN,NaN,1501.0,13.29,1.0,21.33,...,19036.0,19615.0,1679440.0,1,0,0,0,0.0,0.0,0.0
5,73.699997,1,16,0.0,1.25956,NaN,547.3,33.43,0.0,14.67,...,24788.0,21614.0,1640770.0,0,0,0,0,0.0,0.0,0.0
6,80.400002,0,13,0.0,NaN,NaN,<NA>,<NA>,0.5,25.67,...,17963.0,17802.0,1485830.0,1,0,0,0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7105,77.500000,1,18,NaN,NaN,NaN,<NA>,<NA>,1.0,22.00,...,NaN,NaN,NaN,1,0,0,0,0.0,0.0,0.0
7114,65.599998,0,17,NaN,NaN,NaN,<NA>,<NA>,0.0,0.67,...,17860.0,16735.0,1326860.0,0,0,0,0,0.0,1.0,0.0
7117,68.900002,0,18,NaN,NaN,NaN,<NA>,<NA>,0.5,7.00,...,NaN,NaN,NaN,1,0,0,0,0.0,0.0,1.0


In [32]:
X = baseline_adni.drop(columns="DX")
y = baseline_adni["DX"]

N_RUNS = 30
splits = []
for seed in range(N_RUNS):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=seed)
    splits.append((X_tr.index, X_te.index))

In [33]:
# Check proportions 
    # run 0 for overall notion
print(pd.concat({"all": y.value_counts(normalize=True),
                 "train (run 0)": y.loc[splits[0][0]].value_counts(normalize=True),
                 "test (run 0)":  y.loc[splits[0][1]].value_counts(normalize=True)}, axis=1).round(3))

    # major difference between train and test
max_diff = 0
for seed in range(N_RUNS):
    _, _, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    diff = (y_tr.value_counts(normalize=True) - y_te.value_counts(normalize=True)).abs().max()
    max_diff = max(max_diff, diff)

print(f"Major difference between train and test in {N_RUNS} runs: {max_diff:.4f}")

      all  train (run 0)  test (run 0)
DX                                    
1   0.457          0.457         0.456
0   0.372          0.371         0.372
2   0.171          0.171         0.172
Major difference between train and test in 30 runs: 0.0009


<a id="section-3"></a>

# **<font color='#ffffffff'> 3. Preprocessing </font>**

<a id="section-3-1"></a>

## <font color='#0fe073'> 3.1 Missing Values </font>

<a id="section-3-2"></a>

## <font color='#0fe073'> 3.2 Scaling </font>

<a id="section-3-3"></a>

## <font color='#0fe073'> 3.3 Encoding  </font>

<a id="section-3-4"></a>

## <font color='#0fe073'> 3.4  Model  </font>

<a id="section-3-5"></a>

## <font color='#0fe073'> 3.5 Feature Engineering  </font>

<a id="section-3-6"></a>

## <font color='#0fe073'> 3.6 Feature Selection </font>